# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [43]:
import pandas as pd
from IPython.display import display

# Load the dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Clean column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

# Remove unnamed columns
df = df.loc[:, ~df.columns.str.startswith("unnamed")]

# Convert numeric columns
numeric_columns = [
    "customer_lifetime_value",
    "monthly_premium_auto",
    "total_claim_amount",
    "number_of_policies"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

# Clean text columns
text_columns = [
    "response",
    "state",
    "gender",
    "education",
    "policy_type",
    "sales_channel"
]

for column in text_columns:
    if column in df.columns:
        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
        )

# Standardize response values
df["response"] = df["response"].str.title()

print("Cleaned dataset")
print("Rows and columns:", df.shape)
display(df.head())


# 1. Customers with claims below $1,000 who responded Yes
low_claim_yes_customers = df[
    (df["total_claim_amount"] < 1000) &
    (df["response"] == "Yes")
].copy()

print("\n1. Customers with claims below $1,000 who responded Yes")
print("Number of customers:", len(low_claim_yes_customers))

display(
    low_claim_yes_customers[
        [
            "customer",
            "state",
            "response",
            "policy_type",
            "gender",
            "monthly_premium_auto",
            "customer_lifetime_value",
            "total_claim_amount"
        ]
    ].head()
)


# 2. Customers who responded Yes
yes_customers = df[
    df["response"] == "Yes"
].copy()

profitability_summary = (
    yes_customers
    .groupby(
        ["policy_type", "gender"],
        as_index=False
    )
    .agg(
        number_of_customers=("customer", "count"),
        average_monthly_premium=("monthly_premium_auto", "mean"),
        average_customer_lifetime_value=(
            "customer_lifetime_value",
            "mean"
        ),
        average_total_claim_amount=(
            "total_claim_amount",
            "mean"
        )
    )
    .round(2)
)

# Create a simplified value to claim ratio
profitability_summary["clv_to_claim_ratio"] = (
    profitability_summary["average_customer_lifetime_value"] /
    profitability_summary["average_total_claim_amount"]
).round(2)

profitability_summary = (
    profitability_summary
    .sort_values(
        "clv_to_claim_ratio",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n2. Premium, customer lifetime value, and claims")
display(profitability_summary)

highest_clv = profitability_summary.nlargest(
    1,
    "average_customer_lifetime_value"
).iloc[0]

lowest_claim = profitability_summary.nsmallest(
    1,
    "average_total_claim_amount"
).iloc[0]

best_value = profitability_summary.nlargest(
    1,
    "clv_to_claim_ratio"
).iloc[0]

print("\nProfitability and risk conclusion")

print(
    f"The highest average customer lifetime value belongs to "
    f"{highest_clv['gender']} customers with "
    f"{highest_clv['policy_type']} policies, at "
    f"${highest_clv['average_customer_lifetime_value']:,.2f}."
)

print(
    f"The lowest average total claim amount belongs to "
    f"{lowest_claim['gender']} customers with "
    f"{lowest_claim['policy_type']} policies, at "
    f"${lowest_claim['average_total_claim_amount']:,.2f}."
)

print(
    f"The strongest customer value relative to claims belongs to "
    f"{best_value['gender']} customers with "
    f"{best_value['policy_type']} policies. Their CLV to claim "
    f"ratio is {best_value['clv_to_claim_ratio']:.2f}."
)

print(
    "Segments with higher customer lifetime value and lower "
    "average claims may appear more profitable and lower risk. "
    "However, this is a simplified comparison and does not "
    "include operating costs or acquisition costs."
)


# 3. Number of customers by state
state_customer_counts = (
    df.groupby(
        "state",
        as_index=False
    )
    .agg(
        number_of_customers=("customer", "count")
    )
    .sort_values(
        "number_of_customers",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n3. Number of customers by state")
display(state_customer_counts)

states_above_500 = state_customer_counts[
    state_customer_counts["number_of_customers"] > 500
].copy()

print("\nStates with more than 500 customers")
display(states_above_500)

top_state = state_customer_counts.iloc[0]

print(
    f"\n{top_state['state']} has the highest number of customers, "
    f"with {top_state['number_of_customers']:,} customers."
)


# 4. Customer lifetime value by education and gender
clv_by_education_gender = (
    df.groupby(
        ["education", "gender"],
        as_index=False
    )
    .agg(
        maximum_customer_lifetime_value=(
            "customer_lifetime_value",
            "max"
        ),
        minimum_customer_lifetime_value=(
            "customer_lifetime_value",
            "min"
        ),
        median_customer_lifetime_value=(
            "customer_lifetime_value",
            "median"
        )
    )
    .round(2)
    .sort_values(
        "median_customer_lifetime_value",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "\n4. Maximum, minimum, and median customer lifetime value "
    "by education and gender"
)

display(clv_by_education_gender)

highest_median = clv_by_education_gender.nlargest(
    1,
    "median_customer_lifetime_value"
).iloc[0]

lowest_median = clv_by_education_gender.nsmallest(
    1,
    "median_customer_lifetime_value"
).iloc[0]

highest_maximum = clv_by_education_gender.nlargest(
    1,
    "maximum_customer_lifetime_value"
).iloc[0]

print("\nCustomer lifetime value conclusion")

print(
    f"The highest median customer lifetime value belongs to "
    f"{highest_median['gender']} customers with "
    f"{highest_median['education']} education, at "
    f"${highest_median['median_customer_lifetime_value']:,.2f}."
)

print(
    f"The lowest median customer lifetime value belongs to "
    f"{lowest_median['gender']} customers with "
    f"{lowest_median['education']} education, at "
    f"${lowest_median['median_customer_lifetime_value']:,.2f}."
)

print(
    f"The highest maximum customer lifetime value appears among "
    f"{highest_maximum['gender']} customers with "
    f"{highest_maximum['education']} education, at "
    f"${highest_maximum['maximum_customer_lifetime_value']:,.2f}."
)

print(
    "The median is more representative of the typical customer "
    "because it is less affected by extremely high values."
)

Cleaned dataset
Rows and columns: (10910, 25)


,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,48029,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,0,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,22139,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,49078,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,23675,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN



1. Customers with claims below $1,000 who responded Yes
Number of customers: 1399


,customer,state,response,policy_type,gender,monthly_premium_auto,customer_lifetime_value,total_claim_amount
3,XL78013,Oregon,Yes,Corporate Auto,M,97,22332.439460,484.013411
8,FM55990,California,Yes,Personal Auto,M,154,5989.773931,739.200000
15,CW49887,California,Yes,Special Auto,F,114,4626.801093,547.200000
19,NJ54277,California,Yes,Personal Auto,F,94,3746.751625,19.575683
27,MQ68407,Oregon,Yes,Personal Auto,F,111,4376.363592,60.036683



2. Premium, customer lifetime value, and claims


...


States with more than 500 customers


,state,number_of_customers
0,California,3552
1,Oregon,2909
2,Arizona,1937
3,Nevada,993
4,Washington,888



California has the highest number of customers, with 3,552 customers.

4. Maximum, minimum, and median customer lifetime value by education and gender


,education,gender,maximum_customer_lifetime_value,minimum_customer_lifetime_value,median_customer_lifetime_value
0,High School or Below,M,83325.38,1940.98,6286.73
1,High School or Below,F,55277.45,2144.92,6039.55
2,College,M,61134.68,1918.12,6005.85
3,Master,F,51016.07,2417.78,5729.86
4,Bachelor,F,73225.96,1904.00,5640.51
5,College,F,61850.19,1898.68,5623.61
6,Master,M,50568.26,2272.31,5579.10
7,Doctor,M,32677.34,2267.60,5577.67
8,Bachelor,M,67907.27,1898.01,5548.03
9,Doctor,F,44856.11,2395.57,5332.46



Customer lifetime value conclusion
The highest median customer lifetime value belongs to M customers with High School or Below education, at $6,286.73.
The lowest median customer lifetime value belongs to F customers with Doctor education, at $5,332.46.
The highest maximum customer lifetime value appears among M customers with High School or Below education, at $83,325.38.
The median is more representative of the typical customer because it is less affected by extremely high values.


In [44]:
# Convert the date column and create month
df["effective_to_date"] = pd.to_datetime(
    df["effective_to_date"],
    errors="coerce"
)

df["month"] = df["effective_to_date"].dt.month_name()

print("Date and month preview")
display(df[["effective_to_date", "month"]].head())


# Bonus 1
# Count policy records by state and month
state_month_policy_counts = (
    df.dropna(
        subset=["state", "month"]
    )
    .groupby(
        ["state", "month"],
        as_index=False
    )
    .agg(
        policies_sold=("customer", "count")
    )
)

month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

# Table for all states
policies_by_state_month = (
    state_month_policy_counts
    .pivot(
        index="state",
        columns="month",
        values="policies_sold"
    )
    .fillna(0)
    .astype(int)
)

available_months = [
    month
    for month in month_order
    if month in policies_by_state_month.columns
]

policies_by_state_month = policies_by_state_month.reindex(
    columns=available_months
)

print("\nBonus 1: Policies sold by state and month")
display(policies_by_state_month)


# Find top 3 states
top_3_states = (
    df.groupby("state")
      .size()
      .sort_values(ascending=False)
      .head(3)
      .index
      .tolist()
)

top_3_state_month_table = (
    state_month_policy_counts[
        state_month_policy_counts["state"].isin(top_3_states)
    ]
    .pivot(
        index="state",
        columns="month",
        values="policies_sold"
    )
    .fillna(0)
    .astype(int)
)

available_top_months = [
    month
    for month in month_order
    if month in top_3_state_month_table.columns
]

top_3_state_month_table = top_3_state_month_table.reindex(
    index=top_3_states,
    columns=available_top_months
)

print("\nTop 3 states with the highest number of policies sold")
print(top_3_states)

display(top_3_state_month_table)

print(
    f"\nThe top three states are {top_3_states[0]}, "
    f"{top_3_states[1]}, and {top_3_states[2]}."
)


# Bonus 2
# Customer response rate by marketing channel using melt
marketing_long = (
    df[
        [
            "customer",
            "response",
            "sales_channel"
        ]
    ]
    .copy()
    .melt(
        id_vars=[
            "customer",
            "response"
        ],
        value_vars=[
            "sales_channel"
        ],
        var_name="channel_type",
        value_name="marketing_channel"
    )
)

# Handle missing responses safely
marketing_long["responded_yes"] = (
    marketing_long["response"]
    .fillna("No")
    .astype("string")
    .str.strip()
    .str.title()
    .eq("Yes")
    .fillna(False)
    .astype(int)
)

channel_response_rate = (
    marketing_long
    .dropna(subset=["marketing_channel"])
    .groupby(
        "marketing_channel",
        as_index=False
    )
    .agg(
        total_customers=("customer", "count"),
        yes_responses=("responded_yes", "sum"),
        response_rate=("responded_yes", "mean")
    )
)

channel_response_rate["response_rate_percent"] = (
    channel_response_rate["response_rate"] * 100
).round(2)

channel_response_rate = (
    channel_response_rate
    .drop(columns="response_rate")
    .sort_values(
        "response_rate_percent",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nBonus 2: Customer response rate by marketing channel")
display(channel_response_rate)

best_channel = channel_response_rate.iloc[0]
lowest_channel = channel_response_rate.iloc[-1]

print(
    f"\nThe channel with the highest response rate is "
    f"{best_channel['marketing_channel']}, at "
    f"{best_channel['response_rate_percent']:.2f}%."
)

print(
    f"The channel with the lowest response rate is "
    f"{lowest_channel['marketing_channel']}, at "
    f"{lowest_channel['response_rate_percent']:.2f}%."
)

print(
    "The marketing team may prioritize channels with higher "
    "response rates, while also considering campaign costs and "
    "customer lifetime value."
)

Date and month preview


/var/folders/3_/6lgqyskd5v55bylpxy1jbmzr0000gn/T/ipykernel_72067/3207785145.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["effective_to_date"] = pd.to_datetime(


,effective_to_date,month
0,2011-02-18,February
1,2011-01-18,January
2,2011-02-10,February
3,2011-01-11,January
4,2011-01-17,January



Bonus 1: Policies sold by state and month


month,January,February
state,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425



Top 3 states with the highest number of policies sold
['California', 'Oregon', 'Arizona']


month,January,February
state,,
California,1918,1634
Oregon,1565,1344
Arizona,1008,929



The top three states are California, Oregon, and Arizona.

Bonus 2: Customer response rate by marketing channel


,marketing_channel,total_customers,yes_responses,response_rate_percent
0,Agent,4121,742,18.01
1,Web,1626,177,10.89
2,Branch,3022,326,10.79
3,Call Center,2141,221,10.32



The channel with the highest response rate is Agent, at 18.01%.
The channel with the lowest response rate is Call Center, at 10.32%.
The marketing team may prioritize channels with higher response rates, while also considering campaign costs and customer lifetime value.
